In [3]:
import pandas as pd, numpy as np, pickle, os
from sklearn.preprocessing import MinMaxScaler
os.makedirs('../data/processed', exist_ok=True)
train_df = pd.read_pickle('../data/processed/train_df.pkl')
test_df  = pd.read_pickle('../data/processed/test_df.pkl')
print(f"Train: {train_df.shape} | Test: {test_df.shape}")

Train: (125973, 44) | Test: (22544, 44)


In [ ]:
for df in [train_df, test_df]:
    #transforme les grandes valeurs:
    df['src_bytes_log']  = np.log1p(df['src_bytes'])
    df['dst_bytes_log']  = np.log1p(df['dst_bytes'])
    df['duration_log']   = np.log1p(df['duration'])
    df['count_log']      = np.log1p(df['count'])
    
    #Ratios:
    df['bytes_ratio']     = df['src_bytes_log'] - df['dst_bytes_log']
    df['srv_count_ratio'] = df['srv_count'] / (df['count'] + 1)
    df['host_srv_ratio']  = df['dst_host_srv_count'] / (df['dst_host_count'] + 1)
    
    #Comptages:
    df['serror_count'] = df['serror_rate'] * df['count']
    df['rerror_count'] = df['rerror_rate'] * df['count']
    
    #Signatures U2R/R2L (CRITIQUE)
    df['priv_escalation']  = df['root_shell'] + df['su_attempted'] + df['num_root']
    df['shell_ops']        = df['num_shells'] + df['num_file_creations'] + df['num_access_files']
    df['login_fail_ratio'] = df['num_failed_logins'] / (df['num_failed_logins'] + df['logged_in'] + 1)
    #Signatures U2R/R2L:priv_escalation regroupe plusieurs indicateurs liés à l'obtention de privilèges administrateur,
    #  ce qui est souvent associé aux attaques U2R. 
    # login_fail_ratio mesure la proportion d'échecs de connexion, 
    # un comportement fréquent dans les attaques R2L.
    # L'objectif est d'aider le modèle à mieux détecter ces deux familles d'attaques rares.

print(f"Features ajoutées")
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
#Les ratios montrent des proportions.
#  Par exemple, srv_count_ratio permet de voir si les connexions sont réparties sur plusieurs services,
#  ce qui peut indiquer un scan.
#  Les comptages donnent le nombre réel d'événements. 
# Par exemple, serror_count indique le nombre réel d'erreurs de connexion, 
# ce qui peut être un signe d'une attaque DoS ou d'un scan.

Features ajoutées
Train: (125973, 56) | Test: (22544, 56)


In [ ]:
train_services = set(train_df['service'].unique())
test_services = set(test_df['service'].unique())
unknown_services = test_services - train_services

print(f"Services inconnus dans test : {unknown_services}")

test_df['service'] = test_df['service'].apply(
    lambda x: 'unknown' if x in unknown_services else x
)

print(f"Services harmonisés")
# La colonne "service" indique le type de service utilisé par la connexion
# (exemple : http, ftp, telnet, smtp, etc.)
# Certains services peuvent apparaître dans les données de test mais n'existent
# pas dans les données d'entraînement. Comme le modèle ne les a jamais vus,
# il risque de mal les traiter.
# Pour éviter cela, tous les services inconnus sont remplacés par la valeur
# "unknown". Ainsi, le modèle peut les reconnaître comme une catégorie spéciale
# au lieu de les considérer comme une erreur ou une valeur imprévue.

Services inconnus dans test : set()
Services harmonisés


In [ ]:
cat_vars = ['protocol_type','service','flag','land','logged_in','is_host_login','is_guest_login']
n_train = len(train_df)
combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
cat_data = pd.get_dummies(combined[cat_vars]).astype(int)
print(f"cat_data (combined): {cat_data.shape}")
# Les colonnes catégorielles (comme protocol_type, service ou flag)
# contiennent du texte, alors que le modèle ne peut travailler qu'avec
# des valeurs numériques.
# Pour les convertir, on utilise le One-Hot Encoding : chaque valeur
# possible devient une nouvelle colonne contenant uniquement 0 ou 1.
# Par exemple, pour protocol_type, on crée les colonnes
# protocol_type_tcp, protocol_type_udp et protocol_type_icmp.
# Une connexion TCP aura la valeur 1 dans protocol_type_tcp et 0 dans les autres colonnes.
# Le train et le test sont d'abord réunis dans un seul tableau avant
# l'encodage afin de garantir qu'ils possèdent exactement les mêmes
# colonnes après la transformation.

cat_data (combined): (148517, 88)


In [ ]:
exclude = set(cat_vars + ['label','attack_family','binary_label'])
numeric_vars = [c for c in combined.columns if c not in exclude]
numeric_data = combined[numeric_vars].copy()
print(f"numeric_data: {numeric_data.shape}")
# On prépare ici les données finales qui seront envoyées au modèle.
# On garde uniquement les informations utiles pour faire les prédictions;
# - les colonnes numériques (duration, src_bytes, etc.)
# - les colonnes créées par le One-Hot Encoding (0 et 1).
# Les anciennes colonnes texte sont supprimées car elles ont déjà été
# converties en nombres. Les colonnes de réponse (label, attack_family
# et binary_label) sont également retirées, car elles contiennent la
# bonne réponse et le modèle ne doit pas la connaître à l'avance.

numeric_data: (148517, 46)


In [ ]:
numeric_cat_data = pd.concat([numeric_data, cat_data], axis=1)
print(f"numeric_cat_data: {numeric_cat_data.shape}")
toutes les colonnes numériques sont regroupées dans un seul
# tableau. C'est ce tableau final qui sera utilisé pour entraîner
# et tester le modèle.

numeric_cat_data: (148517, 134)


In [ ]:
x_train_full = numeric_cat_data.iloc[:n_train].values
x_test_full  = numeric_cat_data.iloc[n_train:].values
y_train_binary = train_df['binary_label'].values
y_test_binary  = test_df['binary_label'].values
fam2id = {'normal':0,'dos':1,'probe':2,'r2l':3,'u2r':4}
y_train_multi = np.array([fam2id[f] for f in train_df['attack_family']])
y_test_multi  = np.array([fam2id[f] for f in test_df['attack_family']])
print(f"x_train: {x_train_full.shape} | x_test: {x_test_full.shape}")
# On sépare à nouveau le dataset en train et test;
# - les premières lignes = train
# - le reste = test
# .values transforme les données en chiffres simples
# pour que le modèle puisse les utiliser.
# y_train_binary = 0 ou 1;
# 0 = normal, 1 = attaque.
# Les familles d’attaques (texte) sont transformées en chiffres;
# normal=0, dos=1, probe=2, r2l=3, u2r=4.
# Résultat : plus aucun texte, seulement des nombres pour entraîner le modèle.

x_train: (125973, 134) | x_test: (22544, 134)


In [10]:
from imblearn.over_sampling import SMOTE
from collections import Counter

#SMOTE
smote = SMOTE(
    sampling_strategy={
        3: 5000,   # R2L : de 995 → 5000
        4: 2000,   # U2R : de 52 → 2000
    },
    random_state=42,
    k_neighbors=5
)

x_train_full, y_train_multi = smote.fit_resample(x_train_full, y_train_multi)
y_train_binary = (y_train_multi != 0).astype(int)

print(f"Après SMOTE - x_train: {x_train_full.shape}")
print(f"Distribution: {Counter(y_train_multi)}")

Après SMOTE - x_train: (131926, 134)
Distribution: Counter({np.int64(0): 67343, np.int64(1): 45927, np.int64(2): 11656, np.int64(3): 5000, np.int64(4): 2000})


In [ ]:
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train_full).astype(np.float32)
x_test  = scaler.transform(x_test_full).astype(np.float32)
print(f"Normalisé")
# On met toutes les valeurs entre 0 et 1 pour que le modèle travaille mieux.
# Le scaler apprend les min et max uniquement sur le train.
# Puis il transforme le train avec ces valeurs.
# Ensuite, il transforme le test avec les mêmes règles (sans réapprendre).
# Important : on n’utilise jamais le test pour apprendre.

Normalisé


In [12]:
np.save('../data/processed/x_train.npy', x_train)
np.save('../data/processed/x_test.npy', x_test)
np.save('../data/processed/y_train_binary.npy', y_train_binary)
np.save('../data/processed/y_test_binary.npy', y_test_binary)
np.save('../data/processed/y_train_multi.npy', y_train_multi)
np.save('../data/processed/y_test_multi.npy', y_test_multi)
with open('../data/processed/scaler.pkl','wb') as f: pickle.dump(scaler,f)
with open('../data/processed/feature_cols.pkl','wb') as f: pickle.dump(numeric_cat_data.columns.tolist(),f)
print(" Sauvegardé")

 Sauvegardé
